In [4]:
import sys
import torch
import hashlib
import random

from datasets import load_dataset

/Users/akarsh/Documents/projects/fastInfer/StoryForge/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("device:", device)

device: mps


In [22]:
SEED = 2026
TRAIN_STORIES = 100_000
VAL_STORIES = 2_000

In [23]:
def normalized(text):
    return " ".join(text.split())

In [24]:
def story_hash(text):
    return hashlib.sha256(normalized(text).encode()).hexdigest()

In [25]:
def unique_texts(rows, n, excluded=None):
    seen = set() if excluded is None else set(excluded)
    result = []

    for row in rows:
        text = row["text"].strip()
        h = story_hash(text)

        if text and h not in seen:
            result.append(text)
            seen.add(h)

        if len(result) >= n:
            break

    assert len(result) == n, "Not enough distinct stories."
    return result


In [26]:
train_stream = load_dataset(
    "roneneldan/TinyStories",
    split="train",
    streaming=True,
)

val_stream = load_dataset(
    "roneneldan/TinyStories",
    split="validation",
    streaming=True,
)

In [27]:
train_texts = unique_texts(train_stream, TRAIN_STORIES)

In [28]:
train_hashes = {story_hash(text) for text in train_texts}

In [31]:
val_texts = unique_texts(
    val_stream,
    VAL_STORIES,
    excluded=train_hashes,
)

In [1]:
from storyforge.data import story_hash, unique_texts

In [3]:
print(story_hash("Hello world"))

64ec88ca00b268e5ba1a35678a1b5316d212f4f366b2477232534a8aeca37f3c
